In [ ]:
from google.colab import drive
try:
    drive.flush_and_unmount()
except:
    pass
drive.mount('/content/drive', force_remount=True)

!pip install -q tensorflow numpy pandas scikit-learn matplotlib seaborn

import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os, json, glob
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

print(f'✅ TensorFlow: {tf.__version__}')
print(f'✅ GPU: {tf.config.list_physical_devices("GPU")}')

Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive
✅ TensorFlow: 2.19.0
✅ GPU: []


In [ ]:
DRIVE_BASE    = '/content/drive/MyDrive'

# ✅ Same paths you used in Phase 1 & 2
X_PATH        = f'{DRIVE_BASE}/X_transfer.npy'
Y_AGE_PATH    = f'{DRIVE_BASE}/y_age.npy'
Y_GENDER_PATH = f'{DRIVE_BASE}/y_gender.npy'
Y_ACCENT_PATH = f'{DRIVE_BASE}/y_accent.npy'

# Where checkpoints will be saved
CKPT_DIR      = f'{DRIVE_BASE}/voicescope_cnn_bilstm/checkpoints/'
RESULTS_DIR   = f'{DRIVE_BASE}/voicescope_cnn_bilstm/results/'
HISTORY_PATH  = f'{DRIVE_BASE}/voicescope_cnn_bilstm/training_history.json'
FINAL_MODEL   = f'{DRIVE_BASE}/voicescope_cnn_bilstm/final_model.keras'

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

BATCH_SIZE   = 32
EPOCHS       = 50
LR           = 1e-3
TEST_SIZE    = 0.2
RANDOM_STATE = 42

NUM_AGE     = 3
NUM_GENDER  = 2
NUM_ACCENT  = 5

AGE_NAMES    = ['Young (18-30)', 'Middle (30-50)', 'Mature (50+)']
GENDER_NAMES = ['Male', 'Female']
ACCENT_NAMES = ['India', 'US', 'Canada', 'UK', 'Other']

print('✅ Config ready!')

✅ Config ready!


In [ ]:
print('📂 Loading features...')
X        = np.load(X_PATH)
y_age    = np.load(Y_AGE_PATH)
y_gender = np.load(Y_GENDER_PATH)
y_accent = np.load(Y_ACCENT_PATH)

print(f'✅ X shape     : {X.shape}')
print(f'✅ y_age       : {y_age.shape}')
print(f'✅ y_gender    : {y_gender.shape}')
print(f'✅ y_accent    : {y_accent.shape}')

# Safety clip
y_age    = np.clip(y_age,    0, NUM_AGE    - 1)
y_gender = np.clip(y_gender, 0, NUM_GENDER - 1)
y_accent = np.clip(y_accent, 0, NUM_ACCENT - 1)

X_train, X_test, \
y_age_train, y_age_test, \
y_gender_train, y_gender_test, \
y_accent_train, y_accent_test = train_test_split(
    X, y_age, y_gender, y_accent,
    test_size=TEST_SIZE,
    stratify=y_age,
    random_state=RANDOM_STATE
)

print(f'\n✅ Train samples : {X_train.shape[0]}')
print(f'✅ Test samples  : {X_test.shape[0]}')
print(f'Age dist (train) : {np.bincount(y_age_train)}')
print(f'Gender dist      : {np.bincount(y_gender_train)}')
print(f'Accent dist      : {np.bincount(y_accent_train)}')

📂 Loading features...
✅ X shape     : (7510, 128, 120, 1)
✅ y_age       : (7510,)
✅ y_gender    : (7510,)
✅ y_accent    : (7510,)

✅ Train samples : 6008
✅ Test samples  : 1502
Age dist (train) : [1583 2199 2226]
Gender dist      : [3475 2533]
Accent dist      : [2665  300    0  254 2789]


In [ ]:
def build_cnn_bilstm(input_shape=(128, 120, 1),
                     num_age=3, num_gender=2, num_accent=5,
                     dropout_rate=0.4):

    inputs = tf.keras.Input(shape=input_shape, name='mel_spectrogram')

    # CNN Block 1
    x = tf.keras.layers.Conv2D(32, (3,3), padding='same')(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    x = tf.keras.layers.Conv2D(32, (3,3), padding='same')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    x = tf.keras.layers.MaxPooling2D((2,2))(x)   # → (64, 60, 32)
    x = tf.keras.layers.Dropout(0.2)(x)

    # CNN Block 2
    x = tf.keras.layers.Conv2D(64, (3,3), padding='same')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    x = tf.keras.layers.Conv2D(64, (3,3), padding='same')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    x = tf.keras.layers.MaxPooling2D((2,2))(x)   # → (32, 30, 64)
    x = tf.keras.layers.Dropout(0.2)(x)

    # CNN Block 3
    x = tf.keras.layers.Conv2D(128, (3,3), padding='same')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    x = tf.keras.layers.Conv2D(128, (3,3), padding='same')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    x = tf.keras.layers.MaxPooling2D((2,2))(x)   # → (16, 15, 128)
    x = tf.keras.layers.Dropout(0.3)(x)

    # Reshape for BiLSTM: (batch, 16, 15*128=1920)
    freq_dim  = x.shape[1]
    merge_dim = x.shape[2] * x.shape[3]
    x = tf.keras.layers.Reshape((freq_dim, merge_dim))(x)

    # BiLSTM layers
    x = tf.keras.layers.Bidirectional(
            tf.keras.layers.LSTM(128, return_sequences=True), name='bilstm_1')(x)
    x = tf.keras.layers.Dropout(dropout_rate)(x)

    x = tf.keras.layers.Bidirectional(
            tf.keras.layers.LSTM(64, return_sequences=False), name='bilstm_2')(x)
    x = tf.keras.layers.Dropout(dropout_rate)(x)

    # Shared dense
    shared = tf.keras.layers.Dense(256, activation='relu', name='shared_dense')(x)
    shared = tf.keras.layers.BatchNormalization()(shared)
    shared = tf.keras.layers.Dropout(dropout_rate)(shared)

    # Age head
    age_x   = tf.keras.layers.Dense(64, activation='relu')(shared)
    age_out = tf.keras.layers.Dense(num_age, activation='softmax', name='age')(age_x)

    # Gender head
    gen_x   = tf.keras.layers.Dense(32, activation='relu')(shared)
    gen_out = tf.keras.layers.Dense(num_gender, activation='softmax', name='gender')(gen_x)

    # Accent head
    acc_x   = tf.keras.layers.Dense(64, activation='relu')(shared)
    acc_out = tf.keras.layers.Dense(num_accent, activation='softmax', name='accent')(acc_x)

    model = tf.keras.Model(
        inputs=inputs,
        outputs={'age': age_out, 'gender': gen_out, 'accent': acc_out},
        name='CNN_BiLSTM_VoiceScope'
    )
    return model


model = build_cnn_bilstm(
    input_shape=(128, 120, 1),
    num_age=NUM_AGE,
    num_gender=NUM_GENDER,
    num_accent=NUM_ACCENT
)

model.summary()
print(f'\n✅ Total params: {model.count_params():,}')

Model: "CNN_BiLSTM_VoiceScope"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ mel_spectrogram     │ (None, 128, 120,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 128, 120,  │        320 │ mel_spectrogram[… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 128, 120,  │        128 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 128, 120,  │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 128, 120,  │      9,248 │ activation[0][0]  │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 120,  │        128 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 128, 120,  │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 64, 60,    │          0 │ activation_1[0][… │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 64, 60,    │          0 │ max_pooling2d[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 64, 60,    │     18,496 │ dropout[0][0]     │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 60,    │        256 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 64, 60,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 64, 60,    │     36,928 │ activation_2[0][… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 60,    │        256 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 64, 60,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 32, 30,    │          0 │ activation_3[0][… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 32, 30,    │          0 │ max_pooling2d_1[

 Total params: 2,626,506 (10.02 MB)

 Trainable params: 2,625,098 (10.01 MB)

 Non-trainable params: 1,408 (5.50 KB)


✅ Total params: 2,626,506


In [ ]:
def create_focal_loss(alpha=0.3, gamma=2.0):
    def focal_loss_fn(y_true, y_pred):
        ce  = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred, from_logits=False)
        pt  = tf.exp(-ce)
        fl  = alpha * (1 - pt) ** gamma * ce
        return tf.reduce_mean(fl)
    focal_loss_fn.__name__ = 'focal_loss'
    return focal_loss_fn

focal_loss = create_focal_loss(alpha=0.3, gamma=2.0)

model.compile(
    optimizer    = tf.keras.optimizers.Adam(learning_rate=LR),
    loss         = {'age': focal_loss, 'gender': focal_loss, 'accent': focal_loss},
    loss_weights = {'age': 1.0, 'gender': 0.5, 'accent': 1.5},
    metrics      = {'age': ['accuracy'], 'gender': ['accuracy'], 'accent': ['accuracy']}
)

print('✅ Compiled! Focal loss, Adam, multi-output.')

✅ Compiled! Focal loss, Adam, multi-output.


In [ ]:
checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
    filepath         = os.path.join(CKPT_DIR, 'epoch_{epoch:02d}_valloss_{val_loss:.4f}.keras'),
    monitor          = 'val_loss',
    save_best_only   = True,   # ← only saves when val_loss improves
    save_weights_only= False,
    verbose          = 1
)

best_model_cb = tf.keras.callbacks.ModelCheckpoint(
    filepath       = os.path.join(CKPT_DIR, 'BEST_model.keras'),
    monitor        = 'val_loss',
    save_best_only = True,
    verbose        = 1
)

reduce_lr_cb = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1
)

early_stop_cb = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=10, restore_best_weights=True, verbose=1
)

csv_logger_cb = tf.keras.callbacks.CSVLogger(
    filename=os.path.join(RESULTS_DIR, 'training_log.csv'),
    append=True
)

callbacks = [checkpoint_cb, best_model_cb, reduce_lr_cb, early_stop_cb, csv_logger_cb]
print('✅ Callbacks ready!')
print(f'   Checkpoints → {CKPT_DIR}')

✅ Callbacks ready!
   Checkpoints → /content/drive/MyDrive/voicescope_cnn_bilstm/checkpoints/


In [ ]:
print('🚀 Starting fresh training...')

history = model.fit(
    X_train,
    {'age': y_age_train, 'gender': y_gender_train, 'accent': y_accent_train},
    validation_data=(
        X_test,
        {'age': y_age_test, 'gender': y_gender_test, 'accent': y_accent_test}
    ),
    epochs     = EPOCHS,
    batch_size = BATCH_SIZE,
    callbacks  = callbacks,
    verbose    = 1
)

# Save history
with open(HISTORY_PATH, 'w') as f:
    json.dump({k: [float(v) for v in vals] for k, vals in history.history.items()}, f, indent=2)

model.save(FINAL_MODEL)
print(f'✅ Done! Model saved → {FINAL_MODEL}')

🚀 Starting fresh training...
Epoch 1/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - accent_accuracy: 0.3405 - accent_loss: 0.3464 - age_accuracy: 0.3356 - age_loss: 0.2812 - gender_accuracy: 0.5208 - gender_loss: 0.1223 - loss: 0.8619
Epoch 1: val_loss improved from inf to 0.45206, saving model to /content/drive/MyDrive/voicescope_cnn_bilstm/checkpoints/epoch_01_valloss_0.4521.keras

Epoch 1: val_loss improved from inf to 0.45206, saving model to /content/drive/MyDrive/voicescope_cnn_bilstm/checkpoints/BEST_model.keras
188/188 ━━━━━━━━━━━━━━━━━━━━ 1056s 6s/step - accent_accuracy: 0.3409 - accent_loss: 0.3458 - age_accuracy: 0.3356 - age_loss: 0.2809 - gender_accuracy: 0.5208 - gender_loss: 0.1222 - loss: 0.8608 - val_accent_accuracy: 0.4407 - val_accent_loss: 0.1870 - val_age_accuracy: 0.3702 - val_age_loss: 0.1452 - val_gender_accuracy: 0.4294 - val_gender_loss: 0.0526 - val_loss: 0.4521 - learning_rate: 0.0010
Epoch 2/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - accent_accuracy: 0.4583

In [ ]:
ckpt_files = sorted(glob.glob(os.path.join(CKPT_DIR, 'epoch_*.keras')))

if not ckpt_files:
    print('❌ No checkpoints found. Run Cell 7 first.')
else:
    latest_ckpt    = ckpt_files[-1]
    fname          = os.path.basename(latest_ckpt)
    completed_epoch = int(fname.split('_')[1])
    remaining      = EPOCHS - completed_epoch

    print(f'📂 Found {len(ckpt_files)} checkpoint(s)')
    print(f'   Latest : {fname}  (epoch {completed_epoch})')
    print(f'   Remaining epochs: {remaining}')

    if remaining <= 0:
        print('✅ Already finished all epochs!')
    else:
        def create_focal_loss(alpha=0.3, gamma=2.0):
            def focal_loss_fn(y_true, y_pred):
                ce = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred)
                pt = tf.exp(-ce)
                return tf.reduce_mean(alpha * (1 - pt) ** gamma * ce)
            focal_loss_fn.__name__ = 'focal_loss'
            return focal_loss_fn

        model = tf.keras.models.load_model(
            latest_ckpt,
            custom_objects={'focal_loss': create_focal_loss()}
        )
        print(f'✅ Model loaded! Resuming from epoch {completed_epoch} → {EPOCHS}')

        history = model.fit(
            X_train,
            {'age': y_age_train, 'gender': y_gender_train, 'accent': y_accent_train},
            validation_data=(
                X_test,
                {'age': y_age_test, 'gender': y_gender_test, 'accent': y_accent_test}
            ),
            epochs        = EPOCHS,
            initial_epoch = completed_epoch,   # ← key for resuming!
            batch_size    = BATCH_SIZE,
            callbacks     = callbacks,
            verbose       = 1
        )

        # Append history
        existing = {}
        if os.path.exists(HISTORY_PATH):
            with open(HISTORY_PATH) as f:
                existing = json.load(f)
        for k, v in history.history.items():
            existing.setdefault(k, []).extend([float(x) for x in v])
        with open(HISTORY_PATH, 'w') as f:
            json.dump(existing, f, indent=2)

        model.save(FINAL_MODEL)
        print(f'✅ Resumed & saved → {FINAL_MODEL}')

📂 Found 6 checkpoint(s)
   Latest : epoch_09_valloss_0.3952.keras  (epoch 9)
   Remaining epochs: 41
✅ Model loaded! Resuming from epoch 9 → 50
Epoch 10/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - accent_accuracy: 0.4539 - accent_loss: 0.1386 - age_accuracy: 0.3610 - age_loss: 0.1464 - gender_accuracy: 0.5657 - gender_loss: 0.0517 - loss: 0.3802
Epoch 10: val_loss improved from inf to 0.39240, saving model to /content/drive/MyDrive/voicescope_cnn_bilstm/checkpoints/epoch_10_valloss_0.3924.keras

Epoch 10: val_loss improved from inf to 0.39240, saving model to /content/drive/MyDrive/voicescope_cnn_bilstm/checkpoints/BEST_model.keras
188/188 ━━━━━━━━━━━━━━━━━━━━ 978s 5s/step - accent_accuracy: 0.4539 - accent_loss: 0.1387 - age_accuracy: 0.3610 - age_loss: 0.1464 - gender_accuracy: 0.5657 - gender_loss: 0.0517 - loss: 0.3802 - val_accent_accuracy: 0.4654 - val_accent_loss: 0.1479 - val_age_accuracy: 0.3702 - val_age_loss: 0.1448 - val_gender_accuracy: 0.5706 - val_gender_loss: 0.0517 - 

In [ ]:
import tensorflow as tf
import numpy as np
from sklearn.metrics import classification_report
import os

DRIVE_BASE   = '/content/drive/MyDrive'
CKPT_DIR     = f'{DRIVE_BASE}/voicescope_cnn_bilstm/checkpoints/'

AGE_NAMES    = ['Young (18-30)', 'Middle (30-50)', 'Mature (50+)']
GENDER_NAMES = ['Male', 'Female']
ACCENT_NAMES = ['India', 'US', 'Canada', 'UK', 'Other']

# ── Load best model ──────────────────────────────────────────
def create_focal_loss(alpha=0.3, gamma=2.0):
    def focal_loss_fn(y_true, y_pred):
        ce = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred)
        pt = tf.exp(-ce)
        return tf.reduce_mean(alpha * (1 - pt) ** gamma * ce)
    focal_loss_fn.__name__ = 'focal_loss'
    return focal_loss_fn

model = tf.keras.models.load_model(
    os.path.join(CKPT_DIR, 'BEST_model.keras'),
    custom_objects={'focal_loss': create_focal_loss()}
)
print('✅ Best model loaded!')

# ── Predict ──────────────────────────────────────────────────
print('🔮 Predicting...')
preds = model.predict(X_test, verbose=1)

y_pred_age    = np.argmax(preds['age'],    axis=1)
y_pred_gender = np.argmax(preds['gender'], axis=1)
y_pred_accent = np.argmax(preds['accent'], axis=1)

age_acc    = np.mean(y_pred_age    == y_age_test)
gender_acc = np.mean(y_pred_gender == y_gender_test)
accent_acc = np.mean(y_pred_accent == y_accent_test)

print('\n' + '='*50)
print('  CNN-BiLSTM — FINAL ACCURACY (49 epochs)')
print('='*50)
print(f'  👴 Age Accuracy    : {age_acc*100:.2f}%')
print(f'  ♂️♀️ Gender Accuracy : {gender_acc*100:.2f}%')
print(f'  🌍 Accent Accuracy : {accent_acc*100:.2f}%')
print('='*50)

print('\n👴 AGE REPORT:')
print(classification_report(y_age_test, y_pred_age, target_names=AGE_NAMES))

print('\n♂️♀️ GENDER REPORT:')
print(classification_report(y_gender_test, y_pred_gender, target_names=GENDER_NAMES))

print('\n🌍 ACCENT REPORT:')
accent_labels = sorted(np.unique(np.concatenate([y_accent_test, y_pred_accent])))
accent_names_ = [ACCENT_NAMES[i] for i in accent_labels]
print(classification_report(y_accent_test, y_pred_accent,
                             labels=accent_labels, target_names=accent_names_))

✅ Best model loaded!
🔮 Predicting...
47/47 ━━━━━━━━━━━━━━━━━━━━ 57s 1s/step

  CNN-BiLSTM — FINAL ACCURACY (49 epochs)
  👴 Age Accuracy    : 59.12%
  ♂️♀️ Gender Accuracy : 94.41%
  🌍 Accent Accuracy : 67.11%

👴 AGE REPORT:
                precision    recall  f1-score   support

 Young (18-30)       0.50      0.17      0.25       396
Middle (30-50)       0.52      0.71      0.60       550
  Mature (50+)       0.70      0.78      0.74       556

      accuracy                           0.59      1502
     macro avg       0.57      0.55      0.53      1502
  weighted avg       0.58      0.59      0.56      1502


♂️♀️ GENDER REPORT:
              precision    recall  f1-score   support

        Male       0.94      0.96      0.95       857
      Female       0.94      0.92      0.93       645

    accuracy                           0.94      1502
   macro avg       0.94      0.94      0.94      1502
weighted avg       0.94      0.94      0.94      1502


🌍 ACCENT REPORT:
              p

In [ ]:
import glob

# ── See what checkpoints exist ───────────────────────────────
ckpt_files = sorted(glob.glob(os.path.join(CKPT_DIR, '*.keras')))
print(f'📂 Found {len(ckpt_files)} checkpoint(s):')
for f in ckpt_files:
    print(f'   {os.path.basename(f)}')

📂 Found 0 checkpoint(s):
